### **Guilines for Prompting**
#### **Principles of Prompting** (clear != short)

In [12]:
import os, openai
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI
from pprint import pprint


_ = load_dotenv(find_dotenv())
openai.api_key = os.getenv('OPENAI_API_KEY')

llm_model = 'gpt-3.5-turbo'
chat = ChatOpenAI(model = llm_model)

Principle 1: Write clear and specific instructions 
- Tactic 1: Use delimiters to clearly indicate distinct parts of the input. Delimiters can be anything like: `triple quotes/backticks`, `< >`, `<tag> </tag>`, `:`, `---`

In [13]:
text = f'''
You should express what you want a model to do by providing instructions that are as clear and
specific as you can possibly make them. This will guide the model towards the desired output,
and reduce the chances of receiving irrelevant or incorrect responses. Don't confuse writing a
clear prompt with writing a short prompt. In many cases, longer prompts provide more clarity
and context for the model, which can lead to more detailed and relevant outputs.
'''
prompt = f'''
Summarize the text delimited by triple backticks into a single sentence.
```{text}```
'''

response = chat.invoke(prompt)
pprint(response.content)

('Providing clear and specific instructions for a model is essential for '
 'guiding it towards the desired output and reducing the chances of irrelevant '
 'or incorrect responses, regardless of prompt length.')


- Tactic 2: Ask for structured output HTML, JSON

In [17]:
prompt = f'''
Generate a list of three made-up book titles along with their authors and genres. 
Provide them in JSON format with the following keys: book_id, title, author, genre.
'''

response = chat.invoke(prompt)
pprint(response.content)

('```json\n'
 '[\n'
 '    {\n'
 '        "book_id": 1,\n'
 '        "title": "The Midnight Chronicles",\n'
 '        "author": "Eleanor Nightingale",\n'
 '        "genre": "Fantasy"\n'
 '    },\n'
 '    {\n'
 '        "book_id": 2,\n'
 '        "title": "Whispers of the Unseen",\n'
 '        "author": "Julian Blackwood",\n'
 '        "genre": "Mystery"\n'
 '    },\n'
 '    {\n'
 '        "book_id": 3,\n'
 '        "title": "Echoes of the Stars",\n'
 '        "author": "Serena Silvermoon",\n'
 '        "genre": "Science Fiction"\n'
 '    }\n'
 ']\n'
 '```')


- Tactic 3: Ask the model to check whether conditions are satisfied

In [18]:
text_1 = f'''
Making a cup of tea is easy! First, you need to get some water boiling. While that's happening, 
grab a cup and put a tea bag in it. Once the water is hot enough, just pour it over the tea bag. 
Let it sit for a bit so the tea can steep. After a few minutes, take out the tea bag. If you 
like, you can add some sugar or milk to taste. And that's it! You've got yourself a delicious 
cup of tea to enjoy.
'''
prompt = f'''
You will be provided with text delimited by triple quotes. If it contains a sequence of instructions,
re-write those instructions in the following format:

Step 1 - ...
Step 2 - ...

Step N - ...

If the text does not contain a sequence of instructions, then simply write 'No steps provided.'
```{text_1}```
'''

response = chat.invoke(prompt)
pprint(response.content)

('Step 1 - Get some water boiling.\n'
 'Step 2 - Grab a cup and put a tea bag in it.\n'
 'Step 3 - Pour the hot water over the tea bag.\n'
 'Step 4 - Let the tea steep.\n'
 'Step 5 - Remove the tea bag.\n'
 'Step 6 - Add sugar or milk to taste.')


In [ ]:
text_2 = f'''
The sun is shining brightly today, and the birds are singing. It's a beautiful day to go for a 
walk in the park. The flowers are blooming, and the trees are swaying gently in the breeze. People  
are out and about, enjoying the lovely weather. Some are having picnics, while others are playing  
games or simply relaxing on the grass. It's a perfect day to spend time outdoors and appreciate the  
beauty of nature.
'''
prompt = f'''
You will be provided with text delimited by triple quotes. If it contains a sequence of instructions, 
re-write those instructions in the following format:

Step 1 - ...
Step 2 - ...

Step N - ...

If the text does not contain a sequence of instructions, then simply write 'No steps provided.'
```{text_2}```
'''

response = chat.invoke(prompt)
pprint(response.content)

'No steps provided.'


- Tactic 4: Few-shot prompting. Give successful examples of completing tasks. THen ask model to perform the task

In [ ]:
prompt = f'''
Your task is to answer in a consistent style.

<child>: Teach me about patience.

<grandparent>: The river that carves the deepest valley flows from a modest spring; the 
grandest symphony originates from a single note; the most intricate tapestry begins with a solitary thread.

<child>: Teach me about resilience.
'''

response = chat.invoke(prompt)
pprint(response.content)

('<grandparent>: Like a sturdy tree that weathers every storm, resilience is '
 'the ability to bounce back from adversity and overcome challenges with '
 'persistence and determination. Just as the phoenix rises from its ashes, '
 'resilience allows us to rise above setbacks and keep moving forward.')


Principle 2: Give the model time to 'think'
- Tactic 1: Specify the steps required to complete a task

In [ ]:
text = f'''
In a charming village, siblings Jack and Jill set out on a quest to fetch water from a hilltop 
well. As they climbed, singing joyfully, misfortune struck—Jack tripped on a stone and tumbled 
down the hill, with Jill following suit. Though slightly battered, the pair returned home to 
comforting embraces. Despite the mishap, their adventurous spirits remained undimmed, and they 
continued exploring with delight.
'''

# example 1
prompt_1 = f'''
Perform the following actions: 
1 - Summarize the following text delimited by triple backticks with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following keys: french_summary, num_names.

Separate your answers with line breaks.
Text:
```{text}```
'''

response = chat.invoke(prompt_1)
pprint(response.content)

('1 - Jack and Jill go on a quest to fetch water, but encounter misfortune on '
 'their way back home.\n'
 '\n'
 "2 - Jack et Jill partent en quête d'eau, mais rencontrent un accident sur le "
 'chemin du retour.\n'
 '\n'
 '3 - Jack, Jill\n'
 '\n'
 '4 - \n'
 '{\n'
 '  "french_summary": "Jack et Jill partent en quête d\'eau, mais rencontrent '
 'un accident sur le chemin du retour.",\n'
 '  "num_names": 2\n'
 '}')


Ask for output in a specified format

In [23]:
prompt_2 = f'''
Your task is to perform the following actions: 
1 - Summarize the following text delimited by <> with 1 sentence.
2 - Translate the summary into French.
3 - List each name in the French summary.
4 - Output a json object that contains the following keys: french_summary, num_names.

Use the following format:
Text: <text to summarize>
Summary: <summary>
Translation: <summary translation>
Names: <list of names in summary>
Output JSON: <json with original content and num_names>

Text: <{text}>
'''

response = chat.invoke(prompt_2)
pprint(response.content)

('Summary: Jack and Jill, siblings, go on a cheerful quest to fetch water from '
 'a hilltop well, but encounter misfortune along the way. Despite the mishap, '
 'they return home with adventurous spirits intact.\n'
 '\n'
 'Translation: Jack et Jill, deux frères et sœurs, partent joyeusement '
 "chercher de l'eau dans un puits au sommet d'une colline, mais rencontrent "
 'des malheurs en chemin.\n'
 '\n'
 'Names: Jack, Jill\n'
 '\n'
 'Output JSON: \n'
 '{\n'
 '  "french_summary": "Jack et Jill, deux frères et sœurs, partent joyeusement '
 "chercher de l'eau dans un puits au sommet d'une colline, mais rencontrent "
 'des malheurs en chemin.",\n'
 '  "num_names": 2\n'
 '}')


- Tactic 2: Instruct the model to work out its own solution before rushing to a conclusion

In [25]:
prompt = f'''
Determine if the student's solution is correct or not.

Question:
I'm building a solar power installation and I need  help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot
What is the total cost for the first year of operations 
as a function of the number of square feet.

Student's Solution:
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
'''

response = chat.invoke(prompt)
pprint(response.content)

("The student's solution is incorrect. The total cost for the first year of "
 'operations should be calculated as follows:\n'
 '\n'
 'Total cost = Land cost + Solar panel cost + Maintenance cost\n'
 'Total cost = (100 * x) + (250 * x) + (100,000 + (10 * x))\n'
 'Total cost = 100x + 250x + 100,000 + 10x\n'
 'Total cost = 360x + 100,000\n'
 '\n'
 'Therefore, the correct total cost for the first year of operations as a '
 'function of the number of square feet is 360x + 100,000.')


Note that the student's solution is actually not correct. We can fix this by instructing the model to work out its own solution first.

In [27]:
prompt = f'''
Your task is to determine if the student's solution is correct or not.
To solve the problem do the following:
- First, work out your own solution to the problem including the final total. 
- Then compare your solution to the student's solution and evaluate if the student's solution is correct or not. 
Don't decide if the student's solution is correct until you have done the problem yourself.

Use the following format:
Question:
```
question here
```
Student's solution:
```
student's solution here
```
Actual solution:
```
steps to work out the solution and your solution here
```
Is the student's solution the same as actual solution just calculated:
```
yes or no
```
Student grade:
```
correct or incorrect
```

Question:
```
I'm building a solar power installation and I need help working out the financials. 
- Land costs $100 / square foot
- I can buy solar panels for $250 / square foot
- I negotiated a contract for maintenance that will cost me a flat $100k per year, and an additional $10 / square foot
What is the total cost for the first year of operations as a function of the number of square feet.
``` 
Student's solution:
```
Let x be the size of the installation in square feet.
Costs:
1. Land cost: 100x
2. Solar panel cost: 250x
3. Maintenance cost: 100,000 + 100x
Total cost: 100x + 250x + 100,000 + 100x = 450x + 100,000
```
Actual solution:
'''

response = chat.invoke(prompt)
pprint(response.content)

('Total cost = Land cost + Solar panel cost + Maintenance cost\n'
 'Total cost = 100x + 250x + (100,000 + 10x)\n'
 'Total cost = 350x + 100,000 + 10x\n'
 'Total cost = 360x + 100,000\n'
 '\n'
 'The correct total cost for the first year of operations as a function of the '
 'number of square feet is 360x + 100,000.\n'
 '```\n'
 '\n'
 "Is the student's solution the same as actual solution just calculated:\n"
 '```\n'
 'No\n'
 '```\n'
 '\n'
 'Student grade:\n'
 '```\n'
 'Incorrect\n'
 '```')


#### **Model limitations: Hallucinations**
Boie is a real company, the product name is not real.

To reduce hallucinations, first find relevant information, then answer the question based on the relevant information

In [28]:
prompt = f'''
Tell me about AeroGlide UltraSlim Smart Toothbrush by Boie
'''

response = chat.invoke(prompt)
pprint(response.content)

('The AeroGlide UltraSlim Smart Toothbrush by Boie is a cutting-edge electric '
 'toothbrush that offers advanced technology and sleek design for an optimal '
 'brushing experience. This toothbrush features ultra-thin brush heads with '
 'soft, BPA-free bristles that are gentle on gums and effectively clean '
 'teeth. \n'
 '\n'
 'The AeroGlide UltraSlim Smart Toothbrush also comes equipped with smart '
 'technology that helps track your brushing habits and provides real-time '
 'feedback on your technique through a mobile app. The app also allows you to '
 'set personalized brushing goals and reminders to ensure you are maintaining '
 'proper oral hygiene.\n'
 '\n'
 'Additionally, this toothbrush is designed for convenience and portability, '
 'with a slim and lightweight design that makes it easy to take with you '
 'on-the-go. The AeroGlide UltraSlim Smart Toothbrush is rechargeable and '
 'lasts up to 3 weeks on a single charge, making it a convenient and '
 'eco-friendly choice for